# 03 — Feedback Loops

A useful loop does more than repeat. It evaluates an artifact, records actionable feedback, changes evidence or strategy, and tries again under an explicit termination policy.

## Evaluator–optimizer topology

```mermaid
flowchart TD
    accTitle: Feedback Driven Improvement Loop
    accDescr: Generation is evaluated at a quality gate, then either completes, improves under a remaining attempt budget, or stops at fallback.

    generate[Generate] --> evaluate[Evaluate]
    evaluate --> gate{Quality policy}
    gate -->|Pass| complete([END: quality reached])
    gate -->|Fail and budget remains| improve[Improve strategy]
    improve --> generate
    gate -->|Attempts exhausted| fallback([END: fallback])
```

`Generate → Fail → Generate` is retrying. `Generate → Evaluate → Feedback → Change strategy → Generate` is controlled improvement.

## Separate computation from policy

The shared module exposes small node functions. `generate` creates a draft, the evaluator computes score and feedback, and `improve` changes the next strategy. The router alone applies thresholds and attempt limits.

In [6]:
from langgraph.graph import END, START, StateGraph

from graph_engineering.feedback import (
    MAX_ATTEMPTS,
    QUALITY_THRESHOLD,
    FeedbackState,
    build_feedback_graph,
    complete,
    default_score,
    fallback,
    generate,
    improve,
    make_evaluator,
    quality_route,
)

print(f"quality threshold: {QUALITY_THRESHOLD}")
print(f"maximum attempts: {MAX_ATTEMPTS}")

quality threshold: 0.8
maximum attempts: 3


## Inspect one improvement cycle

Direct calls expose the state changes. Feedback is not useful until the optimizer turns it into a different strategy.

In [7]:
initial = {"topic": "bounded graph loops", "attempts": 0, "trace": []}
generated = {**initial, **generate(initial)}
evaluated = {**generated, **make_evaluator(default_score)(generated)}
improvement = improve(evaluated)

assert quality_route(evaluated) == "improve"
assert "feedback" in improvement["strategy"]
print(evaluated)
print(improvement)

{'topic': 'bounded graph loops', 'attempts': 1, 'trace': ['evaluate:0.65'], 'draft': 'Draft 1 about bounded graph loops: state the main claim.', 'score': 0.65, 'feedback': 'Add a concrete limitation and supporting detail.'}
{'strategy': 'revise using feedback: Add a concrete limitation and supporting detail.', 'trace': ['improve']}


## Compile the bounded loop

The business policy has three named outcomes: complete, improve, or fallback. A runtime recursion limit may still be useful as an emergency guard, but it is not the termination policy.

In [8]:
builder = StateGraph(FeedbackState)
builder.add_node("generate", generate)
builder.add_node("evaluate", make_evaluator(default_score))
builder.add_node("improve", improve)
builder.add_node("complete", complete)
builder.add_node("fallback", fallback)

builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")
builder.add_conditional_edges(
    "evaluate",
    quality_route,
    {"complete": "complete", "improve": "improve", "fallback": "fallback"},
)
builder.add_edge("improve", "generate")
builder.add_edge("complete", END)
builder.add_edge("fallback", END)

graph = builder.compile()

## Success termination

The deterministic scorer improves with attempts. The trace should show that the second generation follows feedback rather than blindly repeating the first.

In [9]:
result = graph.invoke({"topic": "bounded graph loops", "attempts": 0, "trace": []})

assert result["score"] >= QUALITY_THRESHOLD
assert result["attempts"] <= MAX_ATTEMPTS
assert result["termination_reason"] == "quality_reached"
print(result["trace"])

['generate:1', 'evaluate:0.65', 'improve', 'generate:2', 'evaluate:0.85', 'complete']


## Exhausted termination

A scorer that never passes lets us test the safety condition. The graph must take the fallback edge at exactly the configured maximum.

In [10]:
persistent_failure_graph = build_feedback_graph(score_fn=lambda _state: 0.1)
failed_result = persistent_failure_graph.invoke(
    {"topic": "persistent low quality", "attempts": 0, "trace": []}
)

assert failed_result["attempts"] == MAX_ATTEMPTS
assert failed_result["termination_reason"] == "attempt_budget_exhausted"
print(failed_result["trace"])

['generate:1', 'evaluate:0.10', 'improve', 'generate:2', 'evaluate:0.10', 'improve', 'generate:3', 'evaluate:0.10', 'fallback']


## Takeaways

- Evaluators compute assessment data; routers apply policy.
- Improvement must change information or strategy.
- Every loop needs success and safety termination.
- Termination reason belongs in observable state.

Next: Notebook 04 runs independent nodes in parallel and merges their updates safely.